<a href="https://colab.research.google.com/github/anawag/pandas-numpy/blob/main/censo_2022.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Import das bases e criação dos dataframes

In [17]:
from google.colab import auth
auth.authenticate_user()

In [18]:
# @title
from google.cloud import bigquery

client = bigquery.Client(project="censo-2022-480100")

alfabetizados = """
SELECT *
FROM `basedosdados.br_ibge_censo_2022.alfabetizacao_grupo_idade_sexo_raca`
"""

esgoto = """
SELECT *
FROM `basedosdados.br_ibge_censo_2022.caracteristica_domicilio_grupo_idade_raca_esgotamento_sanitario`
"""

lixo = """
SELECT *
FROM `basedosdados.br_ibge_censo_2022.caracteristica_domicilio_grupo_idade_raca_destino_lixo`
"""

agua = """
SELECT *
FROM `basedosdados.br_ibge_censo_2022.caracteristica_domicilio_grupo_idade_raca_ligacao_abastecimento_agua`
"""

municipio = """
SELECT *
FROM `basedosdados.br_ibge_censo_2022.municipio`
"""

idade_genero = """
SELECT *
FROM basedosdados.br_ibge_censo_2022.populacao_idade_sexo
"""

alfabetizados_df = client.query(alfabetizados).to_dataframe()
alfabetizados_raw = alfabetizados_df.copy()


esgoto_raw = client.query(esgoto).to_dataframe()
esgoto_raw= esgoto_df.copy()

lixo_raw = client.query(lixo).to_dataframe()
lixo_raw = lixo_df.copy()


agua_raw = client.query(agua).to_dataframe()
agua_raw = agua_df.copy()


municipio_raw = client.query(municipio).to_dataframe()
municipio_raw = municipio_df.copy()


idade_genero_raw = client.query(idade_genero).to_dataframe()
idade_genero_raw = idade_genero_df.copy()



Alfabetizados

In [19]:
# @title
rename_alfabetizados = {
    "cor_raca": "ds_corRaca",
    "sexo": "ds_sexo",
    "grupo_idade": "ds_grupoIdade",
    "alfabetizacao": "ds_alfabetizacao",
    "populacao": "qt_vidasOficial"
}

dtype_alfabetizados = {
    "id_municipio": "string",
    "ds_corRaca": "string",
    "ds_sexo": "string",
    "ds_grupoIdade": "string",
    "ds_alfabetizacao": "string",
}

alfabetizados_raw = (
    alfabetizados_raw
        .rename(columns=rename_alfabetizados)
        .astype(dtype_alfabetizados)
)

nulo_alfabetizados = alfabetizados_raw.columns[alfabetizados_raw.isna().any()].tolist() # identificando a coluna com valores nulos

alfabetizados_raw["qt_vidasOficial"] = alfabetizados_raw["qt_vidasOficial"].fillna(0).astype("int64") # transformando os valores nulos em 0

for col in alfabetizados_raw.columns: # descobrindo dados duplicados
  if alfabetizados_raw[col].duplicated().any():
     print(f"A coluna {col} possui valores duplicados")
  else:
     print(f"A coluna {col} não possui valores duplicados")

id_duplicado_alfa = alfabetizados_raw["id_municipio"].duplicated() # descobrindo quais ids estão duplicados

alfabetizados_raw = alfabetizados_raw.drop_duplicates(subset="id_municipio", keep="first") # mantendo só a primeira vez que o id surge


for col in alfabetizados_raw.columns:
    if alfabetizados_raw[col].dtype == "string":
        alfabetizados_raw[col] = alfabetizados_raw[col].str.upper().str.normalize('NFKD').str.encode(
            'ascii', errors='ignore').str.decode('utf-8') # tranformando em maiúsculas e removendo acentos

print(f"\nTabela Alfabetizados")

print(f"\n {alfabetizados_raw}")

A coluna id_municipio possui valores duplicados
A coluna ds_corRaca possui valores duplicados
A coluna ds_sexo possui valores duplicados
A coluna ds_grupoIdade possui valores duplicados
A coluna ds_alfabetizacao possui valores duplicados
A coluna qt_vidasOficial possui valores duplicados

Tabela Alfabetizados

       id_municipio ds_corRaca   ds_sexo ds_grupoIdade   ds_alfabetizacao  \
0          1100023   INDIGENA    HOMENS  15 A 19 ANOS  NAO ALFABETIZADAS   
1          1100262    AMARELA  MULHERES  15 A 19 ANOS  NAO ALFABETIZADAS   
2          1101005    AMARELA  MULHERES  15 A 19 ANOS  NAO ALFABETIZADAS   
3          1101435    AMARELA  MULHERES  15 A 19 ANOS  NAO ALFABETIZADAS   
5          1100346   INDIGENA  MULHERES  15 A 19 ANOS  NAO ALFABETIZADAS   
...            ...        ...       ...           ...                ...   
48767      3106200    AMARELA    HOMENS  15 A 19 ANOS  NAO ALFABETIZADAS   
49902      5208707   INDIGENA  MULHERES  15 A 19 ANOS  NAO ALFABETIZADAS   
500

In [20]:
# @title
rename_esgoto = {
  "ano": "nr_ano",
  "tipo_esgotamento_sanitario": "tp_esgotamento",
  "grupo_idade": "ds_grupoIdade",
  "cor_raca": "ds_corRaca",
  "populacao": "qt_vidasOficial"
}


dtype_esgoto = {
  "nr_ano" : "string",
  "id_municipio" : "string",
  "tp_esgotamento" : "string",
  "ds_grupoIdade" : "string",
  "ds_corRaca" : "string",
}

esgoto_raw = (
    esgoto_raw
        .rename(columns=rename_esgoto)
        .astype(dtype_esgoto)
)


nulo_esgoto = esgoto_raw.columns[esgoto_raw.isna().any().tolist()]

esgoto_raw["qt_vidasOficial"] = esgoto_raw["qt_vidasOficial"].fillna(0).astype("int64")

for col in esgoto_raw:
  if esgoto_raw[col].duplicated().any():
    print(f"A coluna {col} está sendo duplicada")
  else:
    print(f"A coluna {col} não está sendo duplicada")

id_duplicado_esgoto = esgoto_raw["id_municipio"].duplicated()
print(f"\n {id_duplicado_esgoto}")

esgoto_raw = esgoto_raw.drop_duplicates(subset="id_municipio", keep="first")

for col in esgoto_raw:
  if esgoto_raw[col].dtype == "string":
    esgoto_raw[col] = esgoto_raw[col].str.upper().str.normalize('NFKD').str.encode(
        'ascii', errors='ignore').str.decode('utf-8')

print(f"\nTabela Esgotamento Sanitário")

print(f"\n {esgoto_raw}")

A coluna nr_ano está sendo duplicada
A coluna id_municipio está sendo duplicada
A coluna tp_esgotamento está sendo duplicada
A coluna ds_grupoIdade está sendo duplicada
A coluna ds_corRaca está sendo duplicada
A coluna qt_vidasOficial está sendo duplicada

 0          False
1          False
2          False
3          False
4          False
           ...  
5263645     True
5263646     True
5263647     True
5263648     True
5263649     True
Name: id_municipio, Length: 5263650, dtype: bool

Tabela Esgotamento Sanitário

       nr_ano id_municipio                                   tp_esgotamento  \
0       2022      1100031                        RIO, LAGO, CORREGO OU MAR   
1       2022      1100148  REDE GERAL, REDE PLUVIAL OU FOSSA LIGADA A REDE   
2       2022      1101302                            REDE GERAL OU PLUVIAL   
3       2022      1200252      FOSSA SEPTICA OU FOSSA FILTRO LIGADA A REDE   
4       2022      1200427                                             VALA   
...   

In [21]:
# @title
rename_lixo = {
  "ano": "nr_ano",
  "tipo_destino_lixo": "tp_destinoLixo",
  "grupo_idade": "ds_grupoIdade",
  "cor_raca": "ds_corRaca",
  "populacao": "qt_vidasOficial"
}

dtype_lixo = {
  "nr_ano": "string",
  "id_municipio" : "string",
  "tp_destinoLixo": "string",
  "ds_grupoIdade" : "string",
  "ds_corRaca" : "string"
}

lixo_raw = (
    lixo_raw
      .rename(columns=rename_lixo)
        .astype(dtype_lixo)
  )


nulo_lixo = lixo_raw.columns[lixo_raw.isna().any()].tolist()

lixo_raw["qt_vidasOficial"] = lixo_raw["qt_vidasOficial"].fillna(0).astype("int64")

for col in lixo_raw:
  if lixo_raw[col].duplicated().any():
    print(f"A coluna {col} está sendo duplicada")
  else:
    print(f"A coluna {col} não está sendo duplicada")

id_duplicado_lixo = lixo_raw["id_municipio"].duplicated()
print(f"\n {id_duplicado_lixo}")

lixo_raw = lixo_raw.drop_duplicates(subset="id_municipio", keep="first")

for col in lixo_raw:
  if lixo_raw[col].dtype == "string":
    lixo_raw[col] = lixo_raw[col].str.upper().str.normalize('NFKD').str.encode(
        'ascii', errors='ignore').str.decode('utf-8')


A coluna nr_ano está sendo duplicada
A coluna id_municipio está sendo duplicada
A coluna tp_destinoLixo está sendo duplicada
A coluna ds_grupoIdade está sendo duplicada
A coluna ds_corRaca está sendo duplicada
A coluna qt_vidasOficial está sendo duplicada

 0          False
1          False
2          False
3          False
4          False
           ...  
4093945     True
4093946     True
4093947     True
4093948     True
4093949     True
Name: id_municipio, Length: 4093950, dtype: bool


In [22]:
# @title
rename_agua = {
  "ano": "nr_ano",
  "tipo_ligacao_rede_geral": "tp_ligacao",
  "grupo_idade": "ds_grupoIdade",
  "cor_raca": "ds_corRaca",
  "populacao": "qt_vidasOficial"
}

dtype_agua = {
  "nr_ano": "string",
  "tp_ligacao": "string",
  "ds_grupoIdade": "string",
  "ds_corRaca": "string"
}


agua_raw = (
    agua_raw
     .rename(columns= rename_agua)
        .astype(dtype_agua)
)


nulo_agua = agua_raw.columns[agua_raw.isna().any().tolist()]

agua_raw["qt_vidasOficial"] = agua_raw["qt_vidasOficial"].fillna(0).any().astype("int64")


for col in agua_raw:
  if agua_raw[col].duplicated().any():
    print(f"A coluna {col} está sendo duplicada")
  else:
    print(f"A coluna {col} não está sendo duplicada")


id_duplicado_agua = agua_raw["id_municipio"].duplicated().any()

agua_raw = agua_raw.drop_duplicates(subset="id_municipio", keep="first")

for col in agua_raw:
  if agua_raw[col].dtype == "string":
    agua_raw[col] = agua_raw[col].str.upper().str.normalize('NFKD').str.encode(
        'ascii', errors='ignore').str.decode('utf-8')



A coluna nr_ano está sendo duplicada
A coluna id_municipio está sendo duplicada
A coluna tp_ligacao está sendo duplicada
A coluna ds_grupoIdade está sendo duplicada
A coluna ds_corRaca está sendo duplicada
A coluna qt_vidasOficial está sendo duplicada


In [23]:
rename_municipio = {
  "sigla_uf": "sg_UF",
  "domicilios": "qt_domicilios",
  "populacao": "qt_vidasOficial",
  "area": "qt_area",
  "taxa_alfabetizacao": "vl_percentualAlfabetizacao",
  "idade_mediana": "nr_idadeMediana",
  "razao_sexo": "vl_razaoGenero",
  "indice_envelhecimento": "vl_indiceEnvelhecimento",
  "populacao_indigena": "qt_vidasIndigena",
  "populacao_indigena_terra_indigena": "qt_vidasTerraIndigena",
  "populacao_quilombola": "qt_vidasQuilombola"
}

dtype_municipio = {
    "sg_UF": "string",
    "qt_domicilios": "int64",
    "qt_area": "float64",
    "vl_percentualAlfabetizacao": "float64",
    "nr_idadeMediana": "int64",
    "vl_razaoGenero": "float64",
    "vl_indiceEnvelhecimento": "float64",
    "qt_vidasIndigena": "int64",
    "qt_vidasTerraIndigena": "int64",
    "qt_vidasQuilombola": "int64"
}


municipio_raw = (
    municipio_raw
      .rename(columns= rename_municipio)
        .astype(dtype_municipio)
)


nulo_municipio = municipio_raw.columns[municipio_raw.isna().any().tolist()]

municipio_raw["vl_percentualAlfabetizacao"] = municipio_raw["vl_percentualAlfabetizacao"].map("{:.2f}".format)
municipio_raw["vl_indiceEnvelhecimento"] = municipio_raw["vl_indiceEnvelhecimento"].map("{:.2f}".format)
municipio_raw["vl_razaoGenero"] = municipio_raw["vl_razaoGenero"].map("{:.2f}".format)


for col in municipio_raw:
  if municipio_raw[col].duplicated().any():
    print(f" A {col} está duplicada")
  else:
    print(f" A {col} não está duplicada")

for col in municipio_raw:
  if municipio_raw[col].dtype == ("string"):
    municipio_raw[col] = municipio_raw[col].str.upper().str.normalize('NFKD').str.encode(
        'ascii', errors='ignore').str.decode('utf-8')


 A id_municipio não está duplicada
 A sg_UF está duplicada
 A qt_domicilios está duplicada
 A qt_vidasOficial está duplicada
 A qt_area está duplicada
 A vl_percentualAlfabetizacao está duplicada
 A nr_idadeMediana está duplicada
 A vl_razaoGenero está duplicada
 A vl_indiceEnvelhecimento está duplicada
 A qt_vidasIndigena está duplicada
 A qt_vidasTerraIndigena está duplicada
 A qt_vidasQuilombola está duplicada
 A populacao_quilombola_territorio_quilombola está duplicada


In [24]:
rename_idade_genero = {
    "forma_declaracao_idade": "tp_declaracaoIdade",
    "sexo": "ds_genero",
    "idade": "nm_idade",
    "idade_anos": "nr_idadeAnos",
    "grupo_idade": "ds_grupoIdade",
    "populacao": "qt_vidasOficial"
}

dtype_idade_genero = {
    "tp_declaracaoIdade": "string",
    "ds_genero": "string",
    "nm_idade": "string",
    "ds_grupoIdade": "string"
}

idade_genero_raw = (
    idade_genero_raw
      .rename(columns= rename_idade_genero)
        .astype(dtype_idade_genero)
)


nulo_idade_genero = idade_genero_raw.columns[idade_genero_raw.isna().any().tolist()]

#idade_genero_raw["nr_idadeAnos"] = idade_genero_raw["nr_idadeAnos"].fillna(0).any().astype("float64")
#idade_genero_raw["nr_idadeAnos"] = idade_genero_raw["nr_idadeAnos"].map("{:.2f}".format)

idade_genero_raw["qt_vidasOficial"] = idade_genero_raw["qt_vidasOficial"].fillna(0).any().astype("int64")




print(nulo_idade_genero)


Index(['nr_idadeAnos', 'qt_vidasOficial'], dtype='object')
